In [1]:
import mne

file_path = "/home/filipbds/Documents/Ai_projects/eeg/eeg/MM05/Acquisition 232 Data.cnt"

raw = mne.io.read_raw_cnt(file_path, preload=True)

raw.filter(1., 40.)
raw.notch_filter(50)


Reading 0 ... 2477399  =      0.000 ...  2477.399 secs...
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 3301 samples (3.301 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 

<RawCNT | Acquisition 232 Data.cnt, 69 x 2477400 (2477.4 s), ~1.27 GiB, data loaded>

In [2]:
from scipy.io import loadmat

mat = loadmat("/home/filipbds/Documents/Ai_projects/eeg/eeg/MM05/epoch_inds.mat")
print(mat.keys())

dict_keys(['__header__', '__version__', '__globals__', 'clearing_inds', 'thinking_inds', 'speaking_inds'])


In [28]:
mat['speaking_inds']

array([[array([[13196, 15201]], dtype=uint16),
        array([[20305, 22805]], dtype=uint16),
        array([[27796, 29801]], dtype=uint16),
        array([[34902, 38092]], dtype=uint16),
        array([[43080, 45085]], dtype=uint16),
        array([[50154, 52557]], dtype=uint16),
        array([[57547, 59551]], dtype=uint16),
        array([[64636, 67005]], dtype=int32),
        array([[71962, 73984]], dtype=int32),
        array([[79072, 81086]], dtype=int32),
        array([[86077, 88082]], dtype=int32),
        array([[93169, 95770]], dtype=int32),
        array([[100761, 102766]], dtype=int32),
        array([[107854, 109786]], dtype=int32),
        array([[114776, 116797]], dtype=int32),
        array([[121886, 123918]], dtype=int32),
        array([[128891, 130879]], dtype=int32),
        array([[136001, 138734]], dtype=int32),
        array([[143708, 145730]], dtype=int32),
        array([[150800, 153500]], dtype=int32),
        array([[158492, 160496]], dtype=int32),
        a

In [3]:
canale_relevante = ['FC6', 'FT8', 'C4', 'C3', 'C5', 'T7', 'CP5', 'CP3', 'CP1', 'P3']
raw.pick(canale_relevante)
data = raw.get_data()

In [4]:
import numpy as np

X = np.array([data[:, i[0][0]:i[0][0]+ 5000] for i in mat['thinking_inds'][0]])
Y = np.array([0 for _ in mat['thinking_inds'][0]])


In [5]:
X_s = np.array([data[:, i[0][0]:i[0][0]+ 5000] for i in mat['speaking_inds'][0]])
Y_s = np.array([1 for _ in mat['speaking_inds'][0]])
X = np.concatenate((X, X_s), axis=0)
Y = np.concatenate((Y, Y_s), axis=0)

In [40]:
X.shape

(0,)

Pipeline


In [ ]:
import os
import mne
import numpy as np 

X = np.empty((0, 10, 4000))
Y = np.empty((0,))


for dir in os.listdir("/home/filipbds/Documents/Ai_projects/eeg/eeg/"):
    for file in os.listdir("/home/filipbds/Documents/Ai_projects/eeg/eeg/" + dir):
        if file.endswith(".cnt"):
            raw = mne.io.read_raw_cnt("/home/filipbds/Documents/Ai_projects/eeg/eeg/" + dir + "/" + file, preload=True)
            raw.filter(1., 40.)
            raw.notch_filter(50)
            raw.pick(canale_relevante)
            data = raw.get_data()
            
            mat = loadmat("/home/filipbds/Documents/Ai_projects/eeg/eeg/" + dir + "/epoch_inds.mat")
            
            X_t = np.array([data[:, i[0][0]:i[0][0]+ 4000] for i in mat['thinking_inds'][0]])
            Y_t = np.array([0 for _ in mat['thinking_inds'][0]])
            
            try:
                X_s = np.array([data[:, i[0][0]:i[0][0]+ 4000] for i in mat['speaking_inds'][0]])
                Y_s = np.array([1 for _ in mat['speaking_inds'][0]])
                
                X = np.concatenate((X, X_t, X_s), axis=0)
                Y = np.concatenate((Y, Y_t, Y_s), axis=0)
            except ValueError:
                X = np.concatenate((X, X_t), axis=0)
                Y = np.concatenate((Y, Y_t), axis=0)
            


In [7]:
X.shape

(4322, 10, 4000)

In [8]:
X = X[:, :, ::8]

In [10]:
X.shape

(4322, 10, 500)

In [11]:
Y.shape

(4322,)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch import nn
import torch.functional as F
import torch
from torch.utils.data import DataLoader, TensorDataset

In [17]:
#Normalizez
scalator = StandardScaler()
#din samples, channels, time in samples time channels
X_trans = X.transpose(0,2,1)
X_2d = X_trans.reshape(-1, 10)

In [18]:
X_scalat_2d = scalator.fit_transform(X_2d)
X_scalat = X_scalat_2d.reshape(X.shape[0], X.shape[2], X.shape[1])


In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X_scalat, Y, test_size=.2)
X_train_tensor = torch.FloatTensor(X_train)
Y_train_tensor = torch.FloatTensor(Y_train)
dset = TensorDataset(X_train_tensor, Y_train_tensor)
dataloader = DataLoader(dset, batch_size=32, shuffle=True)
   

array([0., 1.])